### Pilot Exploration 
- Data from Kaggle (https://www.kaggle.com/datasets/ghopkins/nba-injuries-2010-2018)
- Goal: 
1) Collect and evaluate 2019-2020 season data. Identifying duplicate episodes. 
2) Compare against Official NBA data using nba_api.
3) Conclude whether this data source will be adequate for our main injury data source. 

##### Loading Entire Dataset: 

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../Data/Raw/injuries_2010-2020.csv")
df["Date"] = pd.to_datetime(df["Date"])

rows, cols = df.shape
print(f"Dataset Loaded: Rows: {rows}, Columns: {cols}")
print("Full dataset ends:", df["Date"].max())
display(df.head())

pilot_df = df.loc[df["Date"].between("2019-10-22", "2020-08-14")].copy()


Dataset Loaded: Rows: 27105, Columns: 5
Full dataset ends: 2020-10-06 00:00:00


,Date,Team,Acquired,Relinquished,Notes
0,2010-10-03,Bulls,NaN,Carlos Boozer,fractured bone in right pinky finger (out inde...
1,2010-10-06,Pistons,NaN,Jonas Jerebko,torn right Achilles tendon (out indefinitely)
2,2010-10-06,Pistons,NaN,Terrico White,broken fifth metatarsal in right foot (out ind...
3,2010-10-08,Blazers,NaN,Jeff Ayres,torn ACL in right knee (out indefinitely)
4,2010-10-08,Nets,NaN,Troy Murphy,strained lower back (out indefinitely)


##### Loading 2019-2020 Dataframe:

In [3]:
pilot_df.head()
rows, cols = pilot_df.shape
print(f"Dataset Loaded: Rows: {rows}, Columns: {cols}")
print("Season records:", len(pilot_df))
print("\n == NA VALUES ==")
display(pilot_df.isna().sum())
pilot_df.head()

Dataset Loaded: Rows: 2210, Columns: 5
Season records: 2210

 == NA VALUES ==


Date               0
Team               0
Acquired        1307
Relinquished     905
Notes              0
dtype: int64

,Date,Team,Acquired,Relinquished,Notes
24807,2019-10-22,Blazers,NaN,Pau Gasol,left foot injury (DTD)
24808,2019-10-22,Bulls,NaN,Chandler Hutchison,strained left hamstring (DTD)
24809,2019-10-22,Celtics,NaN,Romeo Langford,sore right knee (DTD)
24810,2019-10-22,Celtics,NaN,Tacko Fall,concussion (DTD)
24811,2019-10-22,Clippers,NaN,Paul George,placed on IL with right shoulder injury


##### Loading ankle-sprain data for 19-2020 season: 

In [4]:

# Restore the original season records before looking up returns.
pilot_df = df.loc[
    df["Date"].between("2019-10-22", "2020-08-14")
].copy()

ankle_sprains = pilot_df[
pilot_df["Notes"].str.contains("ankle", case=False, na=False)
    & pilot_df["Notes"].str.contains("sprain", case=False, na=False)
].copy()

ankle_sprains.head(20)

,Date,Team,Acquired,Relinquished,Notes
24812,2019-10-22,Clippers,NaN,Rodney McGruder,placed on IL with sprained right ankle
24825,2019-10-22,Suns,NaN,Ty Jerome,sprained right ankle (out indefinitely)
24838,2019-10-23,Cavaliers,NaN,John Henson,placed on IL with sprained right ankle
24849,2019-10-23,Knicks,NaN,Mitchell Robinson,placed on IL with sprained right ankle
24852,2019-10-23,Mavericks,NaN,Ryan Broekhoff,placed on IL with sprained right ankle
24858,2019-10-23,Rockets,NaN,Michael Frazier II,sprained right ankle (DTD)
24861,2019-10-23,Suns,NaN,Ty Jerome,placed on IL with sprained right ankle
24878,2019-10-24,Warriors,NaN,Alec Burks,placed on IL with sprained right ankle
24879,2019-10-24,Warriors,NaN,Alen Smailagic,sprained right ankle (DTD)
24885,2019-10-25,Jazz,NaN,Bojan Bogdanovic,placed on IL with sprained left ankle


In [5]:
print("Candidate records:", len(ankle_sprains))
display(ankle_sprains)

Candidate records: 131


,Date,Team,Acquired,Relinquished,Notes
24812,2019-10-22,Clippers,NaN,Rodney McGruder,placed on IL with sprained right ankle
24825,2019-10-22,Suns,NaN,Ty Jerome,sprained right ankle (out indefinitely)
24838,2019-10-23,Cavaliers,NaN,John Henson,placed on IL with sprained right ankle
24849,2019-10-23,Knicks,NaN,Mitchell Robinson,placed on IL with sprained right ankle
24852,2019-10-23,Mavericks,NaN,Ryan Broekhoff,placed on IL with sprained right ankle
...,...,...,...,...,...
26869,2020-08-06,Heat,NaN,Goran Dragic,placed on IL with sprained left ankle
26917,2020-08-09,Wizards,NaN,Shabazz Napier,placed on IL with sprained right ankle (out fo...
26927,2020-08-10,Nets,NaN,Jarrett Allen,sprained left ankle (DTD)
26968,2020-08-12,Celtics,NaN,Jayson Tatum,placed on IL with sprained left ankle


##### Identifying repeated players and episodes: 

In [6]:
ankle_sprains["Player"] = ankle_sprains["Relinquished"].str.strip()

player_counts = ankle_sprains["Player"].value_counts()
display(player_counts[player_counts > 1])

Player
Deandre Ayton         4
Daniel Theis          3
Jahlil Okafor         3
Moe Wagner            3
Luka Doncic           3
Ty Jerome             2
John Henson           2
Trae Young            2
Myles Turner          2
Serge Ibaka           2
De'Aaron Fox          2
Landry Shamet         2
Rudy Gobert           2
Romeo Langford        2
Naz Mitrou-Long       2
Dewan Hernandez       2
Glenn Robinson III    2
Nerlens Noel          2
Draymond Green        2
Wendell Carter Jr.    2
Abdel Nader           2
R.J. Barrett          2
Jaylen Brown          2
De'Andre Hunter       2
P.J. Washington       2
Dario Saric           2
Meyers Leonard        2
Mario Hezonja         2
Luke Kornet           2
Ky Bowman             2
Derrick Rose          2
Seth Curry            2
Eric Gordon           2
Name: count, dtype: int64

In [7]:
ankle_sprains["Out Date"] = ankle_sprains["Date"]
ankle_sprains["Cleared Date"] = pd.NaT
ankle_sprains["Injury"] = "ankle sprain"
ankle_sprains["Reports"] = 1
ankle_sprains["episode_id"] = ankle_sprains.index

for index, row in ankle_sprains.iterrows():
    # Get this player's return dates.
    return_dates = pilot_df.loc[
        (pilot_df["Acquired"].str.strip() == row["Player"])
        & pilot_df["Notes"].str.contains(
            "activated from IL|returned to lineup",
            case=False,
            na=False
        ),
        "Date"
    ]

    injury_date = row["Date"]
    same_day_return = (return_dates == injury_date).any()
    if not same_day_return:
        later_returns = return_dates[return_dates > injury_date]
        cleared_date = later_returns.min()

        ankle_sprains.loc[index, "Cleared Date"] = cleared_date


# Reports for the same player with the same return date share an ID.
# Reports without a return date stay separate.
for group, reports in ankle_sprains.groupby(["Player", "Cleared Date"]):
    ankle_sprains.loc[reports.index, "episode_id"] = reports.index[0]

# Combine reports into one row per estimated episode.
pilot_df = ankle_sprains.groupby("episode_id", as_index=False).agg({
    "Team": "first",
    "Player": "first",
    "Out Date": "min",
    "Cleared Date": "first",
    "Injury": "first",
    "Reports": "sum"
})

display(pilot_df)

print("Episodes with multiple reports:")
display(pilot_df[pilot_df["Reports"] > 1])

,episode_id,Team,Player,Out Date,Cleared Date,Injury,Reports
0,24812,Clippers,Rodney McGruder,2019-10-22,2019-10-30,ankle sprain,1
1,24825,Suns,Ty Jerome,2019-10-22,2019-12-02,ankle sprain,2
2,24838,Cavaliers,John Henson,2019-10-23,2019-10-26,ankle sprain,1
3,24849,Knicks,Mitchell Robinson,2019-10-23,2019-10-25,ankle sprain,1
4,24852,Mavericks,Ryan Broekhoff,2019-10-23,2019-10-25,ankle sprain,1
...,...,...,...,...,...,...,...
101,26869,Heat,Goran Dragic,2020-08-06,2020-08-10,ankle sprain,1
102,26917,Wizards,Shabazz Napier,2020-08-09,NaT,ankle sprain,1
103,26927,Nets,Jarrett Allen,2020-08-10,2020-08-12,ankle sprain,1
104,26968,Celtics,Jayson Tatum,2020-08-12,NaT,ankle sprain,1


Episodes with multiple reports:


,episode_id,Team,Player,Out Date,Cleared Date,Injury,Reports
1,24825,Suns,Ty Jerome,2019-10-22,2019-12-02,ankle sprain,2
12,24945,Pacers,Myles Turner,2019-10-31,2019-11-16,ankle sprain,2
16,25044,Raptors,Serge Ibaka,2019-11-09,2019-12-01,ankle sprain,2
17,25061,Kings,De'Aaron Fox,2019-11-11,2019-12-17,ankle sprain,2
19,25078,Clippers,Landry Shamet,2019-11-12,2019-12-14,ankle sprain,2
22,25117,Pelicans,Jahlil Okafor,2019-11-14,2019-11-17,ankle sprain,2
28,25226,Jazz,Rudy Gobert,2019-11-23,2019-11-27,ankle sprain,2
29,25259,Celtics,Romeo Langford,2019-11-25,2019-12-11,ankle sprain,2
36,25528,Wizards,Moe Wagner,2019-12-13,2020-02-02,ankle sprain,2
37,25549,Mavericks,Luka Doncic,2019-12-15,2019-12-26,ankle sprain,2


In [8]:
pilot_df.to_csv(
    "../Data/Processed/injury_episodes_2019_20.csv",
    index=False
)